In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
import xgboost as xgb
from xgboost import XGBRegressor

In [2]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelXG/ModelXG.json")

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.955521583557129, 15.866313934326172, 14.249366760253906, 15.079804420471191, 17.348304748535156, 14.1171293258667, 15.171805381774902, 14.040545463562012, 14.312739372253418, 16.036523818969727, 15.253617286682129, 14.583039283752441, 15.329530715942383, 16.86259651184082, 14.661030769348145, 15.184355735778809, 16.112831115722656, 16.036523818969727, 14.187150001525879, 15.637456893920898, 15.329530715942383, 14.353184700012207, 15.62542724609375, 14.219305992126465, 14.374958038330078, 14.112563133239746, 15.337483406066895, 15.171805381774902, 15.637456893920898, 15.631099700927734, 14.187150001525879, 14.017738342285156, 15.68480396270752, 15.346636772155762, 15.39062213897705, 15.7681884765625, 13.955521583557129, 15.329530715942383, 16.462270736694336, 15.171805381774902, 16.6074275970459, 15.83945083618164, 16.349613189697266, 13.893892288208008, 16.036523818969727, 15.346636772155762, 15.171805381774902, 17.348304748535156, 15.112234115600586, 16.6074275970459, 14.219305992

In [5]:
np.average(y_max_arr)

np.float32(15.314895)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)